# **Stage 1 — Univariate Macro Variable Forecasting**
## Stationary-first AR suite | US Quarterly | IFRS 9 ECL
___
**Purpose.** Forecast the raw macroeconomic series used as regressors in the
Stage 2 PD model, and produce the complete historical + forecast regressor
dataset (`SARIMA_regressors_US_Q.csv`) consumed downstream. This notebook merges
two prior workstreams:

- the **clean, config-first pipeline structure** of `02a_Stage1_TimeSeries` — a
  single output artefact, plug-in model layer, Plotly diagnostics, and a
  derive-from-levels step that keeps growth variables internally consistent; with
- the **methodological depth** of `univariate_all_us` — a stationary-first
  transform battery with ADF escalation, four competing models per series
  (Mean · AR · ARMA · SARMA) all fit on the *same* stationary series so their
  information criteria are comparable, expanding-window time-series CV, a Mean
  "do-nothing" benchmark, and paper-ready model-specification reporting
  (coefficients, IC, Ljung–Box, LaTeX).

| Step | Description | Output |
|------|-------------|--------|
| 0 | Configuration & data loading | — |
| 1 | Pipeline functions (transforms · models · specs · reporting) | — |
| 2 | Stationarity diagnostics (ADF battery + escalation) | `_stationarity.csv` |
| 3 | Univariate evaluation — hold-out forecast **through the 2020–21 gap** | `_summary.csv`, `_forecast_matrix_stationary.csv` |
| 4 | Model specifications & parameter reporting | `_model_specs.{csv,tex}`, `_coefficients.csv`, `_model_report.md` |
| 5 | Forward production forecast (refit on full sample) + level reconstruction | — |
| 6 | Derive transformed variables from forecast levels | — |
| 7 | Assemble complete regressor dataset | **`SARIMA_regressors_US_Q.csv`** |
| 8 | Forecast summary & validation | — |

**References.**
Box & Jenkins (1976); Hyndman & Khandakar (2008, *auto.arima*);
Bank & Eder (2021), *A Review on the PD for IFRS 9*, SSRN 3981339;
Djeundje & Crook (2019), *EJOR* 275(1), 319–333.


### Design notes & methodological decisions (audit trail)
___
Merging the two files forces four design choices. Each is recorded here so a
BDO/PRA reviewer can see the reasoning, not just the result.

1. **Forecasting engine — stationary-first, not `auto_arima(d=None)` on
   transformed levels.** Every series is first mapped to a stationary series by an
   *explicit, theory-driven* transform (log-difference for trending positive
   levels; first difference for rates/indices; none for series already in
   growth/return space), with an ADF check on the training window escalating by
   one further difference only when needed. All four models are then fit on that
   single stationary series, so their AIC/BIC/σ² are directly comparable **within
   a variable**.

2. **Transform is fixed per variable, not selected by AIC across
   log/Box-Cox/Yeo-Johnson.** Comparing `auto_arima` AICs across different
   transforms of the same series (as `02a` did) is not a like-for-like comparison:
   the log-likelihoods are computed on differently-scaled data and omit the change-
   of-variables Jacobian, so the ranking is biased toward whichever transform most
   compresses the variance. This is the same non-comparability that rules out
   BIC-weighting across differencing orders. A theory-driven transform is also
   easier to defend to an auditor than "Box-Cox λ=0.37 won on AIC."

3. **Growth/return variables are *derived* from the level forecasts, not forecast
   independently.** Forecasting `us_gdp_yoy_growth` as its own series ignores its
   definitional link to `us_real_gdp` and duplicates information (see `GROUPS`).
   The 12 base series are forecast; the 9 derived series are reconstructed from
   them (preserving internal consistency). *Breadth is preserved in Step 3, which
   still evaluates every genuine series so the redundancy is visible in the RMSE
   table.*

4. **Engineered/derived columns are dropped before forecasting.** `covid_dummy`
   (constant-zero over the forward window), `delinquency_spline` and
   `us_reer_interp` (interpolation inflates autocorrelation and embeds look-ahead),
   `*_raw`, and pre-built `_L*` lag columns are removed via `DROP_ENGINEERED`;
   lags are rebuilt cleanly in Step 7.

**Known long-horizon behaviour (not a bug).** At 20–24-step horizons a stationary
univariate forecast reverts to its unconditional mean; reconstructed levels
therefore drift smoothly (log-linear for `dlog`, linear for `diff`) rather than
tracking turning points. This is why the **Mean baseline** is reported for every
series: a model that cannot beat Mean is not demonstrating skill. Rate series such
as `us_bond_yield_10y` and `us_vix` mean-revert toward their long-run level rather
than holding the last value.


## **0: Configuration & Data Loading**
___
All user-facing settings live here. To change the variable set, horizon, split
dates, lag structure, or output filenames, edit this cell only — the pipeline
functions in Section 1 are never modified directly.

Data is loaded from the project GitHub raw URL by default (public repo, no local
path required); set `USE_LOCAL=True` to read a local CSV instead.

In [ ]:
import warnings, logging, re
from pathlib import Path
from collections import namedtuple

import numpy as np
import pandas as pd
from scipy import stats

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima

import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
logging.getLogger("pmdarima").setLevel(logging.ERROR)

# ── Data source ───────────────────────────────────────────────────────────────
USE_LOCAL       = False
LOCAL_PATH      = "MacroVariables_US_Q.csv"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection"
DATA_FILE       = "EDA_analytical_US_Q.csv"
GITHUB_TOKEN    = None

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR      = Path("results_stage1_univariate"); OUT_DIR.mkdir(exist_ok=True)
MODEL_PREFIX = "SARIMA"                 # change prefix when swapping model family
OUT_REGRESSORS = f"{MODEL_PREFIX}_regressors_US_Q.csv"

# ── Split, horizon, CV ────────────────────────────────────────────────────────
# 2020-21 is EXCLUDED BY THE SPLIT (US data has actual COVID observations):
# train <=2019Q4, forecast multi-step THROUGH the gap, score on 2022Q1+ in
# stationary space (avoids reconstructing levels across the unobserved window).
TRAIN_END  = "2019-12-31"
TEST_START = "2022-01-01"
N_PERIODS  = 20            # forward production horizon: 5y (2026Q1-2030Q4)
M          = 4             # quarterly seasonality
CV_FOLDS   = 3
CV_HORIZON = 8             # quarters per expanding-window CV fold
AR_MAXLAG  = 4
LJUNG_LAG  = 8            # residual whiteness check lag
# Optional common sample start (e.g. "1990-01-01"); None = all available history.
START_DATE = None

# ── Drop engineered / derived columns so only genuine macro series are modelled ─
DROP_ENGINEERED    = True
ENGINEERED_PATTERN = r"(_L\d+$|spline|interp|dummy|_raw$)"

# ── Base stationarity transform per variable ──────────────────────────────────
#   dlog = log first difference (trending positive levels)
#   diff = first difference       (rates / bounded indices)
#   none = already growth/rate/return  (ADF escalates to +diff if still non-stat.)
TRANSFORM_BASE = {
    "us_real_gdp": "dlog", "us_house_price_idx": "dlog", "us_reer": "dlog",
    "us_oil_price": "dlog", "us_credit": "dlog", "us_sp500_close": "dlog",
    "us_industrial_production": "dlog",
    "us_unemployment": "diff", "us_bond_yield_10y": "diff",
    "us_bond_yield_1y": "diff", "us_consumer_confidence": "diff",
    "us_delinquency_rate": "diff",
    "us_cpi": "none", "us_gdp_yoy_growth": "none",
    "us_credit_qoq_growth": "none", "us_credit_yoy_growth": "none",
    "us_term_spread": "none", "us_vix": "none",
    "us_sp500_log_ret": "none", "us_reer_log_ret": "none",
    "us_oil_log_ret": "none", "us_vix_log_ret": "none",
}

# ── Redundancy groups (feed only ONE representative per group to the PD model) ──
GROUPS = {
    "us_real_gdp": "gdp", "us_gdp_yoy_growth": "gdp",
    "us_credit": "credit", "us_credit_qoq_growth": "credit",
    "us_credit_yoy_growth": "credit",
    "us_sp500_close": "sp500", "us_sp500_log_ret": "sp500",
    "us_reer": "reer", "us_reer_log_ret": "reer",
    "us_oil_price": "oil", "us_oil_log_ret": "oil",
    "us_vix": "vix", "us_vix_log_ret": "vix",
}

# ── Base level/rate series forecast directly (derived vars come from these) ─────
RAW_TO_FORECAST = [
    "us_real_gdp", "us_unemployment", "us_cpi", "us_consumer_confidence",
    "us_bond_yield_10y", "us_credit", "us_sp500_close", "us_vix",
    "us_house_price_idx", "us_industrial_production", "us_oil_price", "us_reer",
]

# ── Derived variables reconstructed from the base forecasts ────────────────────
# output_col -> (source_col, method, periods); method in
# {pct_change_yoy, pct_change_qoq, diff, log_diff}
DERIVED_VARS = {
    "us_gdp_yoy_growth":    ("us_real_gdp",             "pct_change_yoy", 4),
    "us_house_price_yoy":   ("us_house_price_idx",      "pct_change_yoy", 4),
    "us_indprod_yoy":       ("us_industrial_production", "pct_change_yoy", 4),
    "us_oil_yoy":           ("us_oil_price",            "pct_change_yoy", 4),
    "us_reer_diff":         ("us_reer",                 "diff",           1),
    "us_credit_qoq_growth": ("us_credit",               "pct_change_qoq", 1),
    "us_sp500_log_ret":     ("us_sp500_close",          "log_diff",       1),
    "us_vix_log_ret":       ("us_vix",                  "log_diff",       1),
    "us_bond_yield_10y_d1": ("us_bond_yield_10y",       "diff",           1),
}

# ── Lag specification (CCF-optimal lags from EDA Section 5) ────────────────────
LAG_SPEC = {
    "us_house_price_yoy": 3, "us_consumer_confidence": 2, "us_unemployment": 1,
    "us_credit_qoq_growth": 6, "us_gdp_yoy_growth": 0, "us_cpi": 6,
    "us_bond_yield_10y_d1": 2, "us_sp500_log_ret": 4, "us_reer_diff": 1,
    "us_indprod_yoy": 3, "us_oil_yoy": 2, "us_vix_log_ret": 0,
}

# ── Plotly palette / recessions ───────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREEN, GREY = (
    "#1F3864", "#2E75B6", "#C0392B", "#E67E22", "#27AE60", "#7F8C8D")
TEMPLATE = "plotly_white"
RECESSIONS_US = [
    ("1990-07-01", "1991-03-31", "Early 1990s"),
    ("2001-03-01", "2001-12-31", "Dot-com"),
    ("2007-12-01", "2009-06-30", "GFC"),
    ("2020-02-01", "2020-04-30", "COVID"),
]

# ── Loader ────────────────────────────────────────────────────────────────────
def load_data() -> pd.DataFrame:
    if USE_LOCAL:
        df = pd.read_csv(LOCAL_PATH, index_col=0, parse_dates=True)
    else:
        url = f"{GITHUB_RAW_BASE}/{DATA_FILE}"
        if GITHUB_TOKEN:
            import requests, io
            r = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
            r.raise_for_status()
            df = pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
        else:
            df = pd.read_csv(url, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index)
    df = df.asfreq("QE-DEC").sort_index()
    if START_DATE:
        df = df.loc[START_DATE:]
    return df

df_hist = load_data()

# Drop engineered columns up front (genuine macro series only)
if DROP_ENGINEERED:
    keep    = [c for c in df_hist.columns if not re.search(ENGINEERED_PATTERN, c, re.I)]
    dropped = [c for c in df_hist.columns if c not in keep]
    df_hist = df_hist[keep]
else:
    dropped = []

print(f"Loaded {DATA_FILE}" + ("  (LOCAL)" if USE_LOCAL else "  (GitHub)"))
print(f"  Shape : {df_hist.shape}")
print(f"  Range : {df_hist.index.min().date()} -> {df_hist.index.max().date()}")
if dropped:
    print(f"  Dropped {len(dropped)} engineered column(s): {dropped}")
if "us_delinquency_rate" in df_hist.columns:
    t = df_hist["us_delinquency_rate"]
    print(f"  Target: us_delinquency_rate — {t.notna().sum()} obs | "
          f"mean {t.mean():.2f} | min {t.min():.2f} | max {t.max():.2f}")
print(f"\n  Base series to forecast ({len(RAW_TO_FORECAST)}):")
for c in RAW_TO_FORECAST:
    ok = "OK" if c in df_hist.columns else "MISSING"
    print(f"    [{ok:7s}] {c:<28s} NaN={df_hist[c].isna().sum() if c in df_hist else '—'}")


## **1: Pipeline Functions**
___
Core functions, never called directly — they are invoked by the driver cells in
Sections 2–7. Four blocks:

- **Transforms** — `apply_transform`, `stationarise` (ADF-escalated),
  `reconstruct_level` (stationary forecast → level path, validated to machine
  precision in Section 5).
- **Models** — `Mean · AR · ARMA · SARMA`, each returning
  `(length-H forecast, fitted_object)` so its specification can be extracted.
- **Specs** — `model_spec` extracts the estimated stationary-space order, the
  full order folded back onto (log-)levels, the coefficient table (std errors,
  z/t, p-values, significance stars), logLik/AIC/BIC/HQIC/σ², and the Ljung–Box
  whiteness p-value.
- **Reporting** — writers that emit the consolidated CSV/Markdown/LaTeX artefacts.

**Plug-in design:** any alternative model returning `(forecast, fitted)` and a
compatible `model_spec` branch drops straight into the same evaluation and
reporting machinery.

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
def apply_transform(s: pd.Series, kind: str) -> pd.Series:
    if kind == "dlog": return np.log(s).diff()
    if kind == "diff": return s.diff()
    return s.copy()

def stationarise(s: pd.Series, base: str):
    """Apply base transform; escalate by one further difference if ADF (training
    window only) still fails to reject a unit root. Returns (z_full, label)."""
    z = apply_transform(s, base); label = base
    ztr = z.loc[:TRAIN_END].dropna()
    try:
        if len(ztr) > 12 and adfuller(ztr, autolag="AIC")[1] >= 0.05:
            z = z.diff(); label = f"{base}+diff"
    except Exception:
        pass
    return z, label

def transform_diffs(label: str):
    """(uses_log, total_d, has_extra_diff, base) from a transform label."""
    parts = label.split("+"); base = parts[0]
    extra = sum(p == "diff" for p in parts[1:])
    uses_log = (base == "dlog")
    base_d = {"dlog": 1, "diff": 1, "none": 0}.get(base, 1)
    return uses_log, base_d + extra, extra > 0, base

def reconstruct_level(z_future, hist_level: pd.Series, label: str) -> np.ndarray:
    """Integrate a stationary forecast back to the original level scale."""
    _, _, extra, base = transform_diffs(label)
    hl = hist_level.dropna(); z = np.asarray(z_future, float)
    if base == "dlog":
        last_log = np.log(hl.iloc[-1])
        dlog = (np.log(hl).diff().iloc[-1] + np.cumsum(z)) if extra else z
        return np.exp(last_log + np.cumsum(dlog))
    if base == "diff":
        last = hl.iloc[-1]
        d = (hl.diff().iloc[-1] + np.cumsum(z)) if extra else z
        return last + np.cumsum(d)
    # base == "none": series is itself the level/rate
    if extra:
        return hl.iloc[-1] + np.cumsum(z)
    return z


# ── Models (each returns (forecast_array, fitted_object)) ─────────────────────
MeanFit = namedtuple("MeanFit", ["mu", "sigma", "nobs"])

def m_mean(ztr, H):
    y = ztr.dropna()
    return (np.full(H, float(y.mean())),
            MeanFit(float(y.mean()), float(y.std(ddof=1)), int(len(y))))

def m_ar(ztr, H, maxlag=AR_MAXLAG):
    y = ztr.dropna(); best = None
    for p in range(1, maxlag + 1):
        try:
            r = AutoReg(y, lags=p, trend="c").fit()
            if best is None or r.aic < best[0]:
                best = (r.aic, p, r)
        except Exception:
            continue
    if best is None:
        raise RuntimeError("AR failed")
    r = best[2]
    return np.asarray(r.predict(start=len(y), end=len(y) + H - 1)), r

def m_arma(ztr, H):
    mdl = auto_arima(ztr.dropna(), d=0, seasonal=False, stepwise=True,
                     error_action="ignore", suppress_warnings=True,
                     information_criterion="aic")
    return np.asarray(mdl.predict(n_periods=H)), mdl

def m_sarma(ztr, H):
    mdl = auto_arima(ztr.dropna(), d=0, D=0, seasonal=True, m=M, stepwise=True,
                     error_action="ignore", suppress_warnings=True,
                     information_criterion="aic")
    return np.asarray(mdl.predict(n_periods=H)), mdl

MODELS = {"Mean": m_mean, "AR": m_ar, "ARMA": m_arma, "SARMA": m_sarma}

def ts_cv(model_fn, ztr: pd.Series, folds=CV_FOLDS, h=CV_HORIZON) -> float:
    """Expanding-window CV RMSE (mean over folds); inf on total failure."""
    z = ztr.dropna(); n = len(z); start = n - folds * h
    if start < 20: start = max(20, n // 2)
    errs = []
    for i in range(folds):
        tr_end = start + i * h
        if tr_end + h > n or tr_end < 20: continue
        tr, te = z.iloc[:tr_end], z.iloc[tr_end:tr_end + h]
        try:
            fc = np.asarray(model_fn(tr, len(te))[0])
            errs.append(np.sqrt(np.mean((fc[:len(te)] - te.values) ** 2)))
        except Exception:
            continue
    return float(np.mean(errs)) if errs else np.inf

def forecast_index(h):
    return pd.date_range(TRAIN_END, periods=h + 1, freq="QE-DEC")[1:]


# ── Spec-string helpers ───────────────────────────────────────────────────────
def _stars(p):
    if p is None or not np.isfinite(p): return ""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def _ljung_p(resid, lag=LJUNG_LAG):
    try:
        r = pd.Series(np.asarray(resid, float)).dropna()
        if len(r) < lag + 2: lag = max(2, len(r) // 2 - 1)
        return float(acorr_ljungbox(r, lags=[lag], return_df=True)["lb_pvalue"].iloc[-1])
    except Exception:
        return np.nan

def _order_str(order, seasonal=None):
    s = f"({order[0]},{order[1]},{order[2]})"
    if seasonal is not None and any(seasonal[:3]):
        s += f"({seasonal[0]},{seasonal[1]},{seasonal[2]})[{seasonal[3]}]"
    return s

def _seasonal_str(seasonal):
    if seasonal is None or not any(seasonal[:3]): return ""
    return f"({seasonal[0]},{seasonal[1]},{seasonal[2]})[{seasonal[3]}]"

def _sig(x, n=4):
    if x is None or not np.isfinite(x): return np.nan
    if x == 0: return 0.0
    from math import floor, log10
    return round(float(x), -int(floor(log10(abs(x)))) + (n - 1))

def _fmt_num(x, dec=4):
    if x is None or not np.isfinite(x): return "—"
    if x != 0 and abs(x) < 10 ** (-dec): return f"{x:.3g}"
    return f"{x:.{dec}f}"


# ── model_spec: full paper-ready description of a fitted model ─────────────────
def model_spec(name: str, fitted, transform_label="none") -> dict:
    uses_log, d_tr, _, _ = transform_diffs(transform_label)
    spec = {"model": name, "transform": transform_label, "log_link": uses_log,
            "est_order": None, "est_seasonal_order": None, "est_order_str": None,
            "full_order": None, "full_order_str": None, "k_params": None,
            "loglik": np.nan, "aic": np.nan, "bic": np.nan, "hqic": np.nan,
            "sigma2": np.nan, "ljung_box_p": np.nan,
            "coef_table": pd.DataFrame(columns=["coef", "std_err", "stat", "p_value"])}

    if name == "Mean":
        mu, sd, n = float(fitted.mu), float(fitted.sigma), int(fitted.nobs)
        s2 = max(sd ** 2 * (n - 1) / n, 1e-12)
        ll = -0.5 * n * (np.log(2 * np.pi) + np.log(s2) + 1.0)
        se = sd / np.sqrt(n) if n > 0 else np.nan
        z = mu / se if se and np.isfinite(se) and se > 0 else np.nan
        spec.update(est_order=(0, 0, 0), est_order_str="(0,0,0)",
                    full_order=(0, d_tr, 0),
                    full_order_str=_order_str((0, d_tr, 0)) + (" on log-levels" if uses_log else ""),
                    k_params=2, loglik=ll, sigma2=s2,
                    aic=-2 * ll + 4, bic=-2 * ll + np.log(max(n, 1)) * 2,
                    coef_table=pd.DataFrame(
                        {"coef": [mu], "std_err": [se], "stat": [z],
                         "p_value": [2 * (1 - stats.norm.cdf(abs(z))) if np.isfinite(z) else np.nan]},
                        index=["const"]))
        return spec

    if name == "AR":
        r = fitted
        p = len(getattr(r, "ar_lags", []) or []) or max(len(r.params) - 1, 0)
        terms = [str(t).split(".")[-1] if t != "const" else "const" for t in r.params.index]
        ct = pd.DataFrame({"coef": np.asarray(r.params), "std_err": np.asarray(r.bse),
                           "stat": np.asarray(r.tvalues), "p_value": np.asarray(r.pvalues)},
                          index=terms)
        sigma2 = float(getattr(r, "sigma2", np.nan))
        if not np.isfinite(sigma2): sigma2 = float(np.var(np.asarray(r.resid)))
        spec.update(est_order=(p, 0, 0), est_order_str=f"({p},0,0)",
                    full_order=(p, d_tr, 0),
                    full_order_str=_order_str((p, d_tr, 0)) + (" on log-levels" if uses_log else ""),
                    k_params=len(r.params), loglik=float(r.llf), aic=float(r.aic),
                    bic=float(r.bic), hqic=float(r.hqic), sigma2=sigma2,
                    ljung_box_p=_ljung_p(r.resid), coef_table=ct)
        return spec

    # ARMA / SARMA (pmdarima -> SARIMAX)
    res = fitted.arima_res_
    order = tuple(int(x) for x in fitted.order)
    so = tuple(int(x) for x in fitted.seasonal_order)
    names = list(res.param_names); params = np.asarray(res.params, float)
    ct = pd.DataFrame({"coef": params, "std_err": np.asarray(res.bse, float),
                       "stat": np.asarray(res.tvalues, float),
                       "p_value": np.asarray(res.pvalues, float)}, index=names)
    sigma2 = float(params[names.index("sigma2")]) if "sigma2" in names else np.nan
    p, d, q = order
    spec.update(est_order=order, est_seasonal_order=(so if any(so[:3]) else None),
                est_order_str=_order_str(order, so),
                full_order=(p, d + d_tr, q),
                full_order_str=_order_str((p, d + d_tr, q), so) + (" on log-levels" if uses_log else ""),
                k_params=len(params), loglik=float(res.llf), aic=float(res.aic),
                bic=float(res.bic), hqic=float(res.hqic), sigma2=sigma2,
                ljung_box_p=_ljung_p(res.resid), coef_table=ct)
    return spec


# ── Report writers ────────────────────────────────────────────────────────────
def spec_scalar_row(variable, group, spec, extra=None) -> dict:
    def rnd(x, d): return round(float(x), d) if x is not None and np.isfinite(x) else np.nan
    row = {"variable": variable, "group": group, "model": spec["model"],
           "transform": spec["transform"], "est_order": spec["est_order_str"],
           "seasonal_order": _seasonal_str(spec["est_seasonal_order"]),
           "full_order": spec["full_order_str"], "k_params": spec["k_params"],
           "logLik": rnd(spec["loglik"], 3), "AIC": rnd(spec["aic"], 3),
           "BIC": rnd(spec["bic"], 3), "HQIC": rnd(spec["hqic"], 3),
           "sigma2": _sig(spec["sigma2"], 4), "ljung_box_p": rnd(spec["ljung_box_p"], 4)}
    if extra: row.update(extra)
    return row

def coef_long_rows(variable, spec) -> list:
    def rnd(x, d=6): return round(float(x), d) if pd.notna(x) else np.nan
    rows = []
    for term, r in spec["coef_table"].iterrows():
        rows.append({"variable": variable, "model": spec["model"], "term": term,
                     "coef": rnd(r["coef"]), "std_err": rnd(r["std_err"]),
                     "stat": rnd(r["stat"], 4), "p_value": rnd(r["p_value"]),
                     "signif": _stars(r["p_value"])})
    return rows

def _md_coef_table(spec, dec=4) -> str:
    ct = spec["coef_table"]
    if ct.empty: return "_no coefficients_\n"
    out = ["| term | coef | std. err. | stat | p | |", "|---|---:|---:|---:|---:|:--|"]
    for term, r in ct.iterrows():
        co = _fmt_num(r["coef"], dec)
        se = _fmt_num(r["std_err"], dec) if pd.notna(r["std_err"]) else "—"
        st = f"{r['stat']:.2f}" if pd.notna(r["stat"]) else "—"
        pv = f"{r['p_value']:.3f}" if pd.notna(r["p_value"]) else "—"
        out.append(f"| `{term}` | {co} | {se} | {st} | {pv} | {_stars(r['p_value'])} |")
    return "\n".join(out) + "\n"

def render_card(variable, group, transform, chosen_model, chosen_label, spec,
                candidates=None, extra_lines=None) -> str:
    def f(x, d=2): return "—" if x is None or not np.isfinite(x) else f"{x:.{d}f}"
    L = [f"### `{variable}`  ·  group: {group}",
         f"**Transform:** `{transform}` → stationary series  ·  "
         f"**Selected by {chosen_label}:** {chosen_model}  \n",
         f"- Estimated order (stationary space): **{spec['est_order_str']}**  "
         f"\n- Full model order: **{spec['full_order_str']}**  ",
         f"- logLik {f(spec['loglik'])} · AIC {f(spec['aic'])} · BIC {f(spec['bic'])} · "
         f"HQIC {f(spec['hqic'])} · σ² {_fmt_num(spec['sigma2'],4)} · "
         f"Ljung–Box({LJUNG_LAG}) p {f(spec['ljung_box_p'],3)}\n",
         "**Coefficients (stationary space):**\n", _md_coef_table(spec)]
    if extra_lines:
        for ln in extra_lines: L.append(ln + "  ")
        L.append("")
    if candidates:
        L += ["\n**Candidate comparison:**\n",
              "| model | est. order | AIC | BIC | RMSE (stat.) | beats Mean |",
              "|---|---|---:|---:|---:|:--:|"]
        for mname, sp, rmse, beats in candidates:
            eo = sp["est_order_str"] if sp else "—"
            aic = f(sp["aic"]) if sp else "—"; bic = f(sp["bic"]) if sp else "—"
            rm = f"{rmse:.4f}" if (rmse is not None and np.isfinite(rmse)) else "—"
            star = " ←" if mname == chosen_model else ""
            L.append(f"| {mname}{star} | {eo} | {aic} | {bic} | {rm} | {'✓' if beats else ''} |")
    L.append("\n")
    return "\n".join(L)

def _latex_escape(x):
    return (str(x).replace("\\", r"\textbackslash{}").replace("_", r"\_")
            .replace("%", r"\%").replace("&", r"\&").replace("[", r"{[}").replace("]", r"{]}"))

def write_latex_specs(path, df, caption, label, float_formats=None):
    float_formats = float_formats or {}
    cols = list(df.columns)
    aligns = "l" + "".join("r" if df[c].dtype.kind in "fi" else "l" for c in cols)
    lines = [r"\begin{table}[htbp]", r"\centering", rf"\caption{{{caption}}}",
             rf"\label{{{label}}}", rf"\begin{{tabular}}{{{aligns}}}", r"\toprule",
             " & ".join([_latex_escape(df.index.name or "variable")] +
                        [_latex_escape(c) for c in cols]) + r" \\", r"\midrule"]
    for idx, r in df.iterrows():
        cells_ = []
        for c in cols:
            v = r[c]
            if isinstance(v, float):
                fmt = float_formats.get(c, "{:.2f}")
                cells_.append("—" if not np.isfinite(v) else fmt.format(v))
            else:
                cells_.append(_latex_escape(v))
        lines.append(" & ".join([_latex_escape(idx)] + cells_) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]
    Path(path).write_text("\n".join(lines))

def emit_model_reports(out_dir, specs_rows, coef_rows, cards, latex_df,
                       title, fit_note, file_prefix="_"):
    out_dir = Path(out_dir)
    specs_df = pd.DataFrame(specs_rows).set_index("variable")
    specs_df.to_csv(out_dir / f"{file_prefix}model_specs.csv")
    coefs_df = pd.DataFrame(coef_rows)
    coefs_df.to_csv(out_dir / f"{file_prefix}coefficients.csv", index=False)
    with open(out_dir / f"{file_prefix}model_report.md", "w") as fh:
        fh.write(f"# {title}\n\n_{fit_note}_\n\n")
        fh.write("Significance: *** p<0.01, ** p<0.05, * p<0.10. σ² and IC are "
                 "comparable **within** a variable (same stationary series), not "
                 "across variables.\n\n---\n\n")
        for c in cards: fh.write(c + "\n")
    if latex_df is not None and len(latex_df):
        write_latex_specs(out_dir / f"{file_prefix}model_specs.tex", latex_df,
                          caption=title, label="tab:model_specs",
                          float_formats={"sigma2": "{:.3g}", "AIC": "{:.1f}",
                                         "BIC": "{:.1f}", "LB_p": "{:.3f}"})
    print(f"  Reports -> {out_dir}/")
    for f in ["model_specs.csv", "coefficients.csv", "model_report.md", "model_specs.tex"]:
        print(f"    {file_prefix}{f}")
    return specs_df, coefs_df

print("Section 1 functions defined:",
      "transforms · models(", list(MODELS), ") · model_spec · reporting")


## **2: Stationarity Diagnostics**
___
Applies the base transform to every genuine macro series and runs an ADF test on
the **training window only** (`≤2019Q4`). Where the base transform still fails to
reject a unit root, the transform escalates by one further difference. The final
per-variable transform recorded here is the one used everywhere downstream.

Note that verdicts are dataset-dependent: several rate/growth series that are
stationary on the full-history file escalate to a further difference on the 1990+
window, which is exactly why the escalation is data-driven rather than hardcoded.

In [ ]:
# Genuine, forecastable series = those with a defined transform and present in data
FORECASTABLE = [c for c in df_hist.columns if c in TRANSFORM_BASE]

stat_rows = []
for name in FORECASTABLE:
    s = df_hist[name].dropna()
    base = TRANSFORM_BASE[name]
    z_base = apply_transform(s, base).loc[:TRAIN_END].dropna()
    try:
        p_base = adfuller(z_base, autolag="AIC")[1] if len(z_base) > 12 else np.nan
    except Exception:
        p_base = np.nan
    z, label = stationarise(s, base)
    ztr = z.loc[:TRAIN_END].dropna()
    try:
        p_final = adfuller(ztr, autolag="AIC")[1] if len(ztr) > 12 else np.nan
    except Exception:
        p_final = np.nan
    stat_rows.append({
        "variable": name, "group": GROUPS.get(name, name), "base": base,
        "adf_p_base": round(p_base, 4) if np.isfinite(p_base) else np.nan,
        "escalated": label != base, "final_transform": label,
        "adf_p_final": round(p_final, 4) if np.isfinite(p_final) else np.nan,
        "n_train": len(ztr),
    })

stat_df = pd.DataFrame(stat_rows).set_index("variable")
stat_df.to_csv(OUT_DIR / "_stationarity.csv")

print("Stationarity battery (ADF on training window <=2019Q4):\n")
print(stat_df.to_string())
print(f"\n  {int(stat_df['escalated'].sum())}/{len(stat_df)} series escalated by a "
      f"further difference.  Saved -> {OUT_DIR}/_stationarity.csv")


## **3: Univariate Evaluation — Forecast Through the 2020–21 Gap**
___
For **every** genuine series (breadth), the four models are fit on the stationary
training series (`≤2019Q4`), used to forecast multi-step **through** the excluded
COVID window, and scored on `2022Q1+` in stationary space. Model selection uses
expanding-window CV; the Mean baseline is the do-nothing benchmark.

Reaching 2022Q1 from 2019Q4 is a 9-step forecast and 2025Q4 is 24 steps — at that
range a stationary univariate forecast reverts to its mean, so a model that does
not beat **Mean** is measuring distance from the long-run average, not skill.
`RMSE_*` columns are in **stationary space** and comparable within a row only.

In [ ]:
H_EVAL = len(pd.date_range(TRAIN_END, "2025-12-31", freq="QE-DEC")) - 1
print(f"Evaluation horizon H = {H_EVAL} quarters (2020Q1..2025Q4); "
      f"scoring 2022Q1+ in stationary space.\n")

def evaluate_variable(name, s_full, H):
    z, tlabel = stationarise(s_full, TRANSFORM_BASE.get(name, "diff"))
    ztr = z.loc[:TRAIN_END].dropna(); zte = z.loc[TEST_START:].dropna()
    if len(zte) < 2 or len(ztr) < 24:
        return None
    fidx = forecast_index(H); common = fidx.intersection(zte.index)

    cv_scores, test_rmse, fcs, fitted_objs = {}, {}, {}, {}
    for mname, fn in MODELS.items():
        cv_scores[mname] = ts_cv(fn, ztr)
        try:
            fc_arr, fitted = fn(ztr, H)
            fc = pd.Series(np.asarray(fc_arr)[:H], index=fidx)
            fcs[mname] = fc; fitted_objs[mname] = fitted
            test_rmse[mname] = float(np.sqrt(((fc[common] - zte[common]) ** 2).mean()))
        except Exception:
            test_rmse[mname] = np.nan

    specs = {}
    for mname, fitted in fitted_objs.items():
        try: specs[mname] = model_spec(mname, fitted, tlabel)
        except Exception: specs[mname] = None

    valid_cv = {k: v for k, v in cv_scores.items() if np.isfinite(v)}
    cv_best = min(valid_cv, key=valid_cv.get) if valid_cv else "Mean"
    valid_te = {k: v for k, v in test_rmse.items() if np.isfinite(v)}
    test_best = min(valid_te, key=valid_te.get) if valid_te else "Mean"

    row = {"variable": name, "group": GROUPS.get(name, name), "transform": tlabel,
           "n_train": len(ztr), "n_test": len(common),
           "cv_best": cv_best,
           "cv_best_order": (specs.get(cv_best) or {}).get("est_order_str", ""),
           "test_best": test_best,
           "test_best_order": (specs.get(test_best) or {}).get("est_order_str", ""),
           "test_best_full_order": (specs.get(test_best) or {}).get("full_order_str", ""),
           "beats_mean": bool(valid_te.get(test_best, np.inf) < valid_te.get("Mean", np.inf))}
    for m in MODELS:
        row[f"RMSE_{m}"] = round(test_rmse.get(m, np.nan), 6) \
            if np.isfinite(test_rmse.get(m, np.nan)) else np.nan

    payload = {"fcs": fcs, "common": common, "specs": specs, "test_rmse": test_rmse,
               "cv_best": cv_best, "test_best": test_best, "transform": tlabel,
               "group": GROUPS.get(name, name), "ztr": ztr, "zte": zte}
    return row, payload

eval_rows, EVAL = [], {}
for name in FORECASTABLE:
    s = df_hist[name].dropna()
    if s.loc[TEST_START:].dropna().shape[0] == 0:
        print(f"  ~ skip {name}: no 2022+ data"); continue
    out = evaluate_variable(name, s, H_EVAL)
    if out is None:
        print(f"  ~ skip {name}: insufficient data"); continue
    row, pl = out; eval_rows.append(row); EVAL[name] = pl
    flag = "" if row["beats_mean"] else "   <-- does NOT beat Mean"
    print(f"  {name:26s}[{row['transform']:9s}] cv={row['cv_best']:5s}"
          f"{row['cv_best_order']:9s} test={row['test_best']:5s}"
          f"{row['test_best_order']:9s}{flag}")

summary = pd.DataFrame(eval_rows).set_index("variable")
summary.to_csv(OUT_DIR / "_summary.csv")

# Forecast matrix (stationary, cv-best per variable) over the scored window
fc_matrix = {}
for name, pl in EVAL.items():
    cvb = pl["cv_best"]
    if cvb in pl["fcs"]:
        fc_matrix[name] = pl["fcs"][cvb].loc[pl["common"]]
if fc_matrix:
    pd.DataFrame(fc_matrix).to_csv(OUT_DIR / "_forecast_matrix_stationary.csv")

n_beat = int(summary["beats_mean"].sum())
print(f"\n{n_beat}/{len(summary)} variables have a model that beats the Mean baseline.")
cols = ["group", "transform", "n_test", "cv_best", "cv_best_order",
        "test_best", "test_best_order", "beats_mean"] + [f"RMSE_{m}" for m in MODELS]
print("\nSUMMARY (test RMSE in stationary space; Mean = do-nothing benchmark):\n")
print(summary[cols].to_string())


In [ ]:
# Through-gap diagnostic: stationary actual (train tail + test) vs cv-best forecast
plot_names = [n for n in RAW_TO_FORECAST if n in EVAL]
n_cols = 3; n_rows = int(np.ceil(len(plot_names) / n_cols))
fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=[f"{n.replace('us_','').replace('_',' ').title()} "
                    f"[{EVAL[n]['transform']}] · {EVAL[n]['cv_best']}"
                    for n in plot_names],
    vertical_spacing=0.09, horizontal_spacing=0.06)

for i, name in enumerate(plot_names):
    r, c = i // n_cols + 1, i % n_cols + 1
    pl = EVAL[name]; ztr = pl["ztr"]; zte = pl["zte"]; cvb = pl["cv_best"]
    fc = pl["fcs"].get(cvb)
    tail = ztr.iloc[-32:]
    fig.add_trace(go.Scatter(x=tail.index, y=tail.values, mode="lines",
        line=dict(color=NAVY, width=1.3), showlegend=(i == 0),
        name="stationary actual"), row=r, col=c)
    fig.add_trace(go.Scatter(x=zte.index, y=zte.values, mode="lines",
        line=dict(color=NAVY, width=1.3), showlegend=False), row=r, col=c)
    if fc is not None:
        fig.add_trace(go.Scatter(x=fc.index, y=fc.values, mode="lines",
            line=dict(color=RED, width=1.4, dash="dash"), showlegend=(i == 0),
            name="cv-best forecast"), row=r, col=c)
    fig.add_vrect(x0=TRAIN_END, x1=TEST_START, fillcolor=GREY, opacity=0.18,
                  line_width=0, row=r, col=c)

fig.update_layout(
    title=dict(text="Figure 1 — Stationary-space forecast through the 2020–21 gap "
               "(train tail + scored test)<br><sup>Grey band = excluded COVID "
               "window · red dashed = CV-selected model forecast</sup>",
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=n_rows * 240, margin=dict(t=90, b=40, l=40, r=20))
fig.update_annotations(font_size=9)
fig.show()


## **4: Model Specifications & Parameter Reporting**
___
Every model that fit successfully contributes a full specification: estimated
stationary-space order, the full order folded back onto (log-)levels, coefficient
table with standard errors / z-t statistics / p-values / significance stars,
logLik/AIC/BIC/HQIC/σ², and the Ljung–Box residual-whiteness p-value.

Four consolidated, paper-ready artefacts are written: a wide `_model_specs.csv`
(one row per variable × model), a long `_coefficients.csv` (one row per term), a
per-variable `_model_report.md` (model cards with candidate comparison), and a
booktabs `_model_specs.tex` for the LaTeX report.

In [ ]:
specs_rows, coef_rows, cards, latex_records = [], [], [], []

for name, pl in EVAL.items():
    for mname in MODELS:
        sp = pl["specs"].get(mname)
        if sp is None:
            continue
        rmse = pl["test_rmse"].get(mname, np.nan)
        specs_rows.append(spec_scalar_row(name, pl["group"], sp, extra={
            "selected_cv": mname == pl["cv_best"],
            "selected_test": mname == pl["test_best"],
            "RMSE_stationary": round(rmse, 6) if np.isfinite(rmse) else np.nan}))
        coef_rows.extend(coef_long_rows(name, sp))

    chosen = pl["test_best"]
    cand = [(m, pl["specs"].get(m), pl["test_rmse"].get(m),
             (np.isfinite(pl["test_rmse"].get(m, np.inf))
              and pl["test_rmse"].get(m, np.inf) < pl["test_rmse"].get("Mean", np.inf)))
            for m in MODELS]
    cards.append(render_card(name, pl["group"], pl["transform"], chosen,
                             "test RMSE", pl["specs"].get(chosen) or {}, cand))
    csp = pl["specs"].get(chosen) or {}
    latex_records.append({"variable": name, "model": chosen,
        "order": csp.get("est_order_str", "—"),
        "full_order": csp.get("full_order_str", "—"),
        "AIC": csp.get("aic", np.nan), "BIC": csp.get("bic", np.nan),
        "sigma2": csp.get("sigma2", np.nan), "LB_p": csp.get("ljung_box_p", np.nan)})

latex_df = pd.DataFrame(latex_records).set_index("variable") if latex_records else None
specs_df, coefs_df = emit_model_reports(
    OUT_DIR, specs_rows, coef_rows, cards, latex_df,
    title="Stage 1 — Univariate model specifications (selection window <=2019Q4)",
    fit_note=("Models fit on the stationary training series (<=2019Q4); orders, "
              "coefficients and IC are reported in stationary space. 'Full order' "
              "folds the transform's differencing back onto (log-)levels."))

# Headline: chosen model per variable (selected by test RMSE)
show = specs_df[specs_df["selected_test"]].copy()
view_cols = ["group", "model", "transform", "est_order", "full_order",
             "AIC", "BIC", "sigma2", "ljung_box_p", "RMSE_stationary"]
print("\nChosen model per variable (selected by test RMSE):\n")
print(show[view_cols].to_string())


## **5: Forward Production Forecast (refit on full sample)**
___
The evaluation in Section 3 measured skill on a hold-out. For the **production**
regressor set there is no gap to forecast through: each base series' CV-selected
model is refit on the **entire** available sample (through the last observed
quarter) and projected `N_PERIODS` (20 quarters) forward.

Forecasts are produced in stationary space and integrated back to levels by
`reconstruct_level`. The round-trip is validated below: transforming the history
and reconstructing it must return the original series to machine precision.

In [ ]:
# ── Round-trip validation of level reconstruction (audit check) ───────────────
print("Level reconstruction round-trip (max |reconstructed - actual| over last 12 obs):")
max_err_all = 0.0
for name in RAW_TO_FORECAST:
    if name not in EVAL: continue
    s = df_hist[name].dropna()
    z, label = stationarise(s, TRANSFORM_BASE[name])
    k = 12; hist = s.iloc[:-k]; truth = s.iloc[-k:]
    z_true = z.reindex(truth.index).values
    rec = reconstruct_level(z_true, hist, label)
    err = float(np.nanmax(np.abs(rec - truth.values)))
    max_err_all = max(max_err_all, err)
    print(f"  {name:26s}[{label:9s}] max_abs_err = {err:.2e}")
assert max_err_all < 1e-6, "Reconstruction failed machine-precision check!"
print(f"\n  PASS — worst-case error {max_err_all:.2e} < 1e-6\n")

# ── Forward index ─────────────────────────────────────────────────────────────
last_date    = df_hist.index[-1]
forecast_idx = pd.date_range(start=last_date + pd.DateOffset(months=3),
                             periods=N_PERIODS, freq="QE")

# ── Refit CV-selected model on full sample; forecast; reconstruct to level ─────
forecast_raw = pd.DataFrame(index=forecast_idx)
fwd_specs_rows, fwd_coef_rows, fwd_cards, fwd_latex = [], [], [], []
print("Forward forecast (model refit on full sample):")
for name in RAW_TO_FORECAST:
    if name not in EVAL:
        print(f"  ~ skip {name}: not evaluated"); continue
    s = df_hist[name].dropna()
    z, label = stationarise(s, TRANSFORM_BASE[name])
    z_full = z.dropna()
    mname = EVAL[name]["cv_best"]
    z_fc, fitted = MODELS[mname](z_full, N_PERIODS)
    level = reconstruct_level(z_fc, s, label)
    forecast_raw[name] = pd.Series(level, index=forecast_idx)

    # Re-extract spec on the full-sample refit (production parameters)
    try:
        sp = model_spec(mname, fitted, label)
        fwd_specs_rows.append(spec_scalar_row(name, GROUPS.get(name, name), sp,
            extra={"selected_cv": True, "selected_test": False}))
        fwd_coef_rows.extend(coef_long_rows(name, sp))
        fwd_cards.append(render_card(name, GROUPS.get(name, name), label, mname,
                                     "CV (full-sample refit)", sp))
        fwd_latex.append({"variable": name, "model": mname,
            "order": sp.get("est_order_str", "—"),
            "full_order": sp.get("full_order_str", "—"),
            "AIC": sp.get("aic", np.nan), "BIC": sp.get("bic", np.nan),
            "sigma2": sp.get("sigma2", np.nan), "LB_p": sp.get("ljung_box_p", np.nan)})
    except Exception:
        sp = None
    print(f"  {name:26s}[{label:9s}] {mname:5s} "
          f"last={s.iloc[-1]:>12.3f}  fc[0]={level[0]:>12.3f}  fc[-1]={level[-1]:>12.3f}")

# Emit forward (production) specification artefacts under a distinct prefix
fwd_latex_df = pd.DataFrame(fwd_latex).set_index("variable") if fwd_latex else None
emit_model_reports(OUT_DIR, fwd_specs_rows, fwd_coef_rows, fwd_cards, fwd_latex_df,
    title="Stage 1 — Forward (production) model specifications (full-sample refit)",
    fit_note=("Production forecasts: the CV-selected model per series refit on the "
              "full sample and projected 20 quarters. Parameters differ from the "
              "hold-out fit because the estimation window now includes 2020+."),
    file_prefix="_forward_")
print(f"\n  forecast_raw (levels) shape: {forecast_raw.shape}")


## **6: Derive Transformed Variables**
___
Growth rates, log returns and first differences are reconstructed **from the base
forecasts** rather than forecast independently — this preserves each derived
series' definitional link to its parent and avoids feeding the PD model redundant,
mutually-inconsistent variants (cf. `GROUPS`). YoY/QoQ calculations need
historical context, so the historical base series is concatenated with the
forecast period before each transformation, then the forecast slice is taken.

In [ ]:
def build_derived(combined, source_col, method, periods, idx):
    if method in ("pct_change_yoy", "pct_change_qoq"):
        return (combined[source_col].pct_change(periods) * 100).loc[idx]
    if method == "diff":
        return combined[source_col].diff(periods).loc[idx]
    if method == "log_diff":
        return np.log(combined[source_col]).diff(periods).loc[idx]
    raise ValueError(f"Unknown method: {method}")

combined = pd.concat([df_hist[RAW_TO_FORECAST], forecast_raw])
for out_col, (src, method, per) in DERIVED_VARS.items():
    if src in combined.columns:
        forecast_raw[out_col] = build_derived(combined, src, method, per, forecast_idx)
    else:
        print(f"  {src} missing — {out_col} skipped")

print(f"Derived variables constructed: {list(DERIVED_VARS.keys())}")
print(f"forecast_raw shape: {forecast_raw.shape}")
nan_cols = forecast_raw.isna().sum()
nan_cols = nan_cols[nan_cols > 0]
print("  No NaN values." if nan_cols.empty else f"  NaN: {nan_cols.to_dict()}")

# ── Figure 2 — all base forecasts ─────────────────────────────────────────────
n_cols = 3; n_rows = int(np.ceil(len(RAW_TO_FORECAST) / n_cols))
fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=[c.replace("us_", "").replace("_", " ").title() for c in RAW_TO_FORECAST],
    vertical_spacing=0.08, horizontal_spacing=0.06)
for i, col in enumerate(RAW_TO_FORECAST):
    r, c = i // n_cols + 1, i % n_cols + 1
    h = df_hist[col].dropna()
    fig.add_trace(go.Scatter(x=h.index[-40:], y=h.values[-40:], mode="lines",
        line=dict(color=NAVY, width=1.5), showlegend=(i == 0), name="Historical"),
        row=r, col=c)
    if col in forecast_raw:
        fc = forecast_raw[col]
        fig.add_trace(go.Scatter(x=fc.index, y=fc.values, mode="lines",
            line=dict(color=RED, width=1.5, dash="dash"),
            showlegend=(i == 0), name="Forecast"), row=r, col=c)
fig.update_layout(
    title=dict(text="Figure 2 — Forward forecasts: all base series (2026–2030)"
               "<br><sup>Last 10 years of history · red dashed = "
               "stationary-first forecast reconstructed to levels</sup>",
               font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=n_rows * 250, margin=dict(t=80, b=40, l=40, r=20))
fig.update_annotations(font_size=9)
fig.show()


## **7: Assemble Complete Regressor Dataset**
___
Concatenates the full historical dataset with the forecast-period series, then
recomputes all lagged columns on the **combined** series so that lag shifts bridge
the history/forecast boundary correctly (e.g. `us_house_price_yoy_L3` at 2026-Q1
is the 2025-Q2 value, which lives in history).

**Output:** `SARIMA_regressors_US_Q.csv` — the single deliverable of this
notebook and the direct input for `Stage2_PD_Modelling`.

In [ ]:
# Concatenate history + forecast (derived cols exist only in the forecast block)
combined_full = pd.concat([df_hist, forecast_raw], axis=0)
combined_full = combined_full.loc[:, ~combined_full.columns.duplicated()]

# Recompute lagged columns on the combined series
for var, lag in LAG_SPEC.items():
    if var in combined_full.columns:
        combined_full[f"{var}_L{lag}"] = combined_full[var].shift(lag)
    else:
        print(f"  {var} not in combined dataset — lag column skipped")

out_path = OUT_DIR / OUT_REGRESSORS
combined_full.to_csv(out_path)
print(f"Saved {OUT_REGRESSORS}  {combined_full.shape}")
print(f"  Path            : {out_path}")
print(f"  Historical rows : {len(df_hist)}")
print(f"  Forecast rows   : {len(forecast_raw)}")
print(f"  Total columns   : {combined_full.shape[1]}")

# Validate: significant lagged regressors over the forecast period
sig_lag_cols = [f"{v}_L{l}" for v, l in LAG_SPEC.items()
                if v in ("us_house_price_yoy", "us_consumer_confidence",
                         "us_unemployment", "us_credit_qoq_growth",
                         "us_gdp_yoy_growth", "us_cpi")
                and f"{v}_L{l}" in combined_full.columns]
print("\nForecast period — Stage 2 significant regressors:")
print(combined_full.loc[forecast_idx, sig_lag_cols].round(4).to_string())
nan_check = combined_full.loc[forecast_idx, sig_lag_cols].isna().sum()
print("\n" + ("✓ No NaN in significant regressors during forecast period."
              if not nan_check.any() else
              f"NaN in forecast-period regressors:\n{nan_check[nan_check > 0]}"))


## **8: Forecast Summary & Validation**
___
Sanity checks and a summary plot of the CCF-significant predictors that enter the
Stage 2 PD regression at their optimal lags. Checks: economically plausible paths
(no explosive trajectories), flat-line detection (a flat path indicates a
random-walk / pure mean-reversion — acceptable but flagged), and no NaN in the
significant regressor columns.

In [ ]:
SIG_VARS = {
    "us_house_price_yoy":     "House Price YoY (%)",
    "us_consumer_confidence": "Consumer Confidence",
    "us_unemployment":        "Unemployment (%)",
    "us_credit_qoq_growth":   "Credit QoQ Growth (%)",
    "us_gdp_yoy_growth":      "GDP YoY Growth (%)",
    "us_cpi":                 "CPI / Inflation",
}
fig = make_subplots(rows=2, cols=3, subplot_titles=list(SIG_VARS.values()),
                    vertical_spacing=0.12, horizontal_spacing=0.07)
for i, (var, label) in enumerate(SIG_VARS.items()):
    r, c = i // 3 + 1, i % 3 + 1
    h = df_hist[var].dropna() if var in df_hist.columns else pd.Series(dtype=float)
    fc = forecast_raw[var] if var in forecast_raw.columns else None
    if len(h):
        cutoff = h.index[-1] - pd.DateOffset(years=10)
        hp = h[h.index >= cutoff]
        for s, e, _ in RECESSIONS_US:
            if pd.Timestamp(s) >= cutoff:
                fig.add_vrect(x0=s, x1=e, fillcolor="lightgrey", opacity=0.3,
                              layer="below", line_width=0, row=r, col=c)
        fig.add_trace(go.Scatter(x=hp.index, y=hp.values, mode="lines",
            line=dict(color=NAVY, width=1.5), showlegend=(i == 0),
            name="Historical"), row=r, col=c)
    if fc is not None:
        fig.add_trace(go.Scatter(x=fc.index, y=fc.values, mode="lines",
            line=dict(color=RED, width=1.8, dash="dash"), showlegend=(i == 0),
            name="Forecast"), row=r, col=c)
fig.update_layout(
    title=dict(text="Figure 3 — Forecasts: CCF-significant predictors (2026–2030)"
               "<br><sup>Grey = recessions · these series enter Stage 2 at their "
               "CCF-optimal lags</sup>", font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=100, b=40, l=50, r=20))
fig.update_annotations(font_size=10)
fig.show()

# ── Summary table + flat-line detection ───────────────────────────────────────
print("\nForecast summary — significant predictors:")
print(f'  {"Variable":<26s} {"Hist last":>10s} {"FC mean":>10s} '
      f'{"FC min":>10s} {"FC max":>10s} {"Flat?":>6s}')
print("  " + "-" * 76)
for var in SIG_VARS:
    if var not in forecast_raw.columns: continue
    hl = df_hist[var].dropna().iloc[-1] if var in df_hist.columns else np.nan
    fc = forecast_raw[var].dropna()
    flat = "Yes" if fc.std() < 1e-3 else "no"
    print(f"  {var:<26s} {hl:>10.3f} {fc.mean():>10.3f} "
          f"{fc.min():>10.3f} {fc.max():>10.3f} {flat:>6s}")

print(f"\n  Deliverable : {OUT_REGRESSORS}")
print(f"  Artefacts   : _stationarity.csv · _summary.csv · "
      f"_forecast_matrix_stationary.csv")
print(f"                _model_specs.{{csv,tex}} · _coefficients.csv · _model_report.md")
print(f"                _forward_model_specs.{{csv,tex}} · _forward_coefficients.csv · "
      f"_forward_model_report.md")
